# 2.7 FlashAttention V1：完整的算法与它的遗憾

2.5 节给出的分块伪代码，其实已经是“现代视角”——循环结构采用的是后来 V2 的形态。本节按历史顺序回到 2022 年，看 FA1 论文的原始算法：它靠两件武器解决了访存问题——**Tiling 分块计算**（前向、反向通用）与**反向重计算**（2.5 节第五节：只存 O(n) 统计量，反向分块重算 S、P，详见彼处），已经是大胜；但在**工程视角**上留下两个遗憾，正是 V2（2.8 节）手术的靶子。

论文：Dao et al., *FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness*, NeurIPS 2022。

## 一、FA1 的原始算法：外层 KV 块、内层 Q 块

![V1 与 V2 的循环结构对比](images/fa_v1_v2_loops.svg)

FA1 论文的 Algorithm 1（单头、无掩码的骨架，记号沿用 2.5 节；本节先只看左半图）：

```text
把 Q 切成 Tr 个行块（每块 M 行），K/V 切成 Tc 个列块（每块 B 行）
for j = 1 … Tc:                          # 外层：KV 块
    把 Kj、Vj 搬入片上
    for i = 1 … Tr:                      # 内层：Q 块
        S   = Qi · Kjᵀ / √d
        m̃   = rowmax(S)
        P̃   = exp(S − m̃)
        l̃   = rowsum(P̃)
        m_new = max(m_i, m̃)
        l_new = e^{m_i − m_new}·l_i + e^{m̃ − m_new}·l̃
        # ↓ 注意这一步：O_i 始终被维护成「已归一化」状态
        O_i  ← ( e^{m_i − m_new}·l_i·O_i + e^{m̃ − m_new}·P̃·V_j ) / l_new
```

与 2.5 节伪代码逐行对照，差异集中在最后两行：

| | FA1（本节） | 2.5 节（V2 结构） |
|--|--|--|
| 外层循环 | KV 块 | Q 块 |
| O 的形态 | **每步都除以 l，保持已归一化** | 未归一化累加（acc），收尾统一除 |
| 每个块对上 O 的逐元素操作 | 3 趟乘 + 1 趟除 | 1 趟乘 |
| exp 的基准 | 本块局部 m̃，随后二次修正 | 直接用当前全局 m_new |

## 二、FA1 的 IO 直觉：为什么这么排循环

外层 KV 的好处在 IO 分析上很直观：**一个 KV 块载入片上后，服务所有 Q 块**——KV 每块只从 HBM 读一次。代价是：每个 Q 块的三个状态 (m_i, l_i, O_i) 必须跨整个外层循环**驻留**片上。

- 序列短（几千 token）时，所有 Q 块的状态连同 O 矩阵都能放进片上 SRAM，论文算法可以直接落地；
- 序列长到放不下时，工程实现只能退化为“每个线程块负责一个 Q 块、内层循环扫 KV”——**概念算法与工程实现开始分家**。

FA1 的 GPU kernel 正是这样实现的：**并行维度只有 (batch, head)**，每个线程块拿到自己的一份 Q 块，循环扫完所有 KV 块。这埋下了第二个遗憾（第四节）。


### 搬运账单：把「O 频繁读写 HBM」算成数字

按 FA1 的循环结构逐项记账（Tr = n/M 个 Q 块，Tc = n/B 个 KV 块）：

| 数据 | 搬运次数 | 总流量 |
|--|--|--|
| K、V | 外层载入，各 **1 次** | = K、V 自身数据量 |
| Q、O、l、m | 每个外层轮次都要**读回片上**，共 Tc 轮 | = (Q+O+l+m) 数据量 **× Tc** |
| O 的中间结果 | 每个块对算完**写回 HBM**、下轮再读回累加 | ≈ O 数据量 × Tc |

看清楚这笔账的结构：K、V 是「好公民」（只读一次）；**Q、O 和两个状态向量被外层循环拖着反复横穿 HBM**——这正是「O 的更新与 Q 绑定」的直接代价。V2 交换循环后 Q 只读 1 次、O 只写 1 次（2.8 节）。

### 每一步算在哪：Cube / Vector 分工

把 FA1 的内层循环体逐行标注到执行单元（昇腾视角，GPU 上 Tensor Core / CUDA core 同构）：

| 计算步骤 | 执行单元 |
|--|--|
| S = Q·Kᵀ（矩阵乘） | **Cube** |
| m̃ = rowmax(S)、P̃ = exp(S−m̃)、l̃ = rowsum(P̃) | **Vector** |
| m_new = max(m, m̃)、l_new（含修正因子 α 的两趟乘） | **Vector** |
| P̃·V（矩阵乘） | **Cube** |
| O 的重标定（×α、×l、÷l_new） | **Vector** |

真实的工程实现还会精打细算片上内存：`S − m̃` 与 `P̃` 共用同一块 Buffer（先减后 exp，原地覆盖）；修正因子的中间量与 `m_new` 共享地址。这类「地址复用」技巧在第 6 章核内切分中会反复出现。

## 三、遗憾一：每步归一化，制造大量 non-matmul 运算

GPU 上 matmul（Tensor Core）的效率远高于逐元素运算；NPU 上同理——**Cube 单元的矩阵乘与 Vector 单元的逐元素运算速率相差悬殊**（第 1 章 2.5 节）。FA2 论文引用了一条经验法则：**每搬运 1 字节片上存储，应配上约 100 FLOPs 的计算**，否则算力抵不过搬运。

而 FA1 的更新式在每个块对上对 O 做「×α、×l、÷l_new」三趟逐元素操作——这些运算：

- 用不了 Tensor Core / Cube 单元（GPU 上走 CUDA core，昇腾上落在 Vector 单元）；
- 数量与块对总数 (Tr × Tc) 成正比，**随序列长度平方增长**。

## 四、遗憾二：并行度只有 B × H

FA1 只在 (batch, head) 两个维度并行。看今天最常见的负载——**长序列、小 batch**（batch=1 的长文推理，或 GQA 下的 decode）：

B=1、H=8 时，108 个 SM 只亮 8 个，占用率 7%（2.8 节实验精确量化）。

短序列大 batch 的训练场景 FA1 吃得满；但 LLM 时代的 decode 与长文推理，任务数远小于核心数——**大量算力闲置**。

## 五、先肯定成绩：FA1 已经大胜

| 指标 | 标准 Attention | FA1 |
|--|--|--|
| HBM 访存 | Θ(nd + n²) | Θ(n²d²/M)（M 为块高） |
| 额外显存 | O(n²) | O(n) |
| 数值 | — | **精确**（非近似） |
| GPT-2 端到端训练 | 基准 | 快约 3×（论文口径，对比当时 MLPerf 纪录） |

FA1 让“训练 64k 上下文”从不可能变成可能。它的问题不是“慢”，而是“**没有跑满硬件**”——论文实测 FA1 在 A100 上只达到理论峰值的 25%~40%。把这部分补回来的，就是 2.8 节的 V2。

## 六、实验：亲手数一数两种风格的逐元素运算

把 FA1 风格（外层 KV、每步归一化）与 V2 风格（外层 Q、收尾统一归一化）都写成 numpy：对拍正确性，并**统计 O 上的逐元素乘除次数**。

In [1]:
import numpy as np
np.random.seed(0)

n, d, M, B = 32, 16, 8, 8          # 小规模便于检查
Q = np.random.randn(n, d)
K = np.random.randn(n, d)
V = np.random.randn(n, d)

def ref_attention(Q, K, V):
    S = Q @ K.T / np.sqrt(Q.shape[-1])
    P = np.exp(S - S.max(-1, keepdims=True))
    return (P / P.sum(-1, keepdims=True)) @ V

def flash_v1(Q, K, V, M, B):
    \"\"\"FA1 论文算法：外层 KV 块、内层 Q 块；O 每步保持已归一化。\"\"\"
    n, d = Q.shape
    O = np.zeros_like(Q)
    m = np.full(n, -np.inf)
    l = np.zeros(n)
    ops = {'O上的逐元素乘法': 0, 'O上的逐元素除法': 0}
    for j0 in range(0, n, B):                       # 外层：KV 块
        Kj, Vj = K[j0:j0 + B], V[j0:j0 + B]
        for i0 in range(0, n, M):                   # 内层：Q 块
            Qi = Q[i0:i0 + M]
            S = Qi @ Kj.T / np.sqrt(d)
            m_blk = m[i0:i0 + M]
            m_new = np.maximum(m_blk, S.max(-1))
            alpha = np.exp(m_blk - m_new)           # m_blk=-inf 时自然为 0
            P = np.exp(S - m_new[:, None])
            l_blk = l[i0:i0 + M]
            l_new = alpha * l_blk + P.sum(-1)
            rows = slice(i0, i0 + M)
            # O ← (α·l·O + P·V) / l_new：3 趟逐元素乘 + 1 趟除
            acc = alpha[:, None] * (l_blk[:, None] * O[rows]) + P @ Vj
            O[rows] = acc / l_new[:, None]
            ops['O上的逐元素乘法'] += 2 * M * d      # α 一趟、l 一趟
            ops['O上的逐元素除法'] += M * d
            m[rows], l[rows] = m_new, l_new
    return O, ops

def flash_v2(Q, K, V, M, B):
    \"\"\"FA2 风格（即 2.5 节伪代码）：外层 Q 块、内层 KV 块；O 未归一化，收尾统一除 l。\"\"\"
    n, d = Q.shape
    O = np.zeros_like(Q)
    m = np.full(n, -np.inf)
    l = np.zeros(n)
    ops = {'O上的逐元素乘法': 0, 'O上的逐元素除法': 0}
    for i0 in range(0, n, M):                       # 外层：Q 块
        rows = slice(i0, i0 + M)
        Qi = Q[i0:i0 + M]
        for j0 in range(0, n, B):                   # 内层：KV 块
            Kj, Vj = K[j0:j0 + B], V[j0:j0 + B]
            S = Qi @ Kj.T / np.sqrt(d)
            m_blk = m[rows]
            m_new = np.maximum(m_blk, S.max(-1))
            alpha = np.exp(m_blk - m_new)
            P = np.exp(S - m_new[:, None])
            l_new = alpha * l[rows] + P.sum(-1)
            # O ← α·O + P·V：只 1 趟逐元素乘；不除 l
            O[rows] = alpha[:, None] * O[rows] + P @ Vj
            ops['O上的逐元素乘法'] += M * d
            m[rows], l[rows] = m_new, l_new
        O[rows] = O[rows] / l[rows][:, None]         # 收尾：每个 Q 块只除一次
        ops['O上的逐元素除法'] += M * d
    return O, ops

O_ref = ref_attention(Q, K, V)
O_v1, ops1 = flash_v1(Q, K, V, M, B)
O_v2, ops2 = flash_v2(Q, K, V, M, B)

print('正确性对拍（与标准 Attention 比）：')
print('  FA1 风格  最大误差 =', np.abs(O_v1 - O_ref).max())
print('  FA2 风格  最大误差 =', np.abs(O_v2 - O_ref).max())
print()
print('O 上的逐元素乘/除运算次数（n=32, M=B=8，共 4×4=16 个块对）：')
print(f"  {'':10s}{'逐元素乘法':>12s}{'逐元素除法':>12s}")
print(f"  {'FA1 风格':10s}{ops1['O上的逐元素乘法']:>12d}{ops1['O上的逐元素除法']:>12d}")
print(f"  {'FA2 风格':10s}{ops2['O上的逐元素乘法']:>12d}{ops2['O上的逐元素除法']:>12d}")
print(f"  除法次数是 V2 的 {ops1['O上的逐元素除法'] / ops2['O上的逐元素除法']:.0f} 倍，乘法多 1/3")

正确性对拍（与标准 Attention 比）：
  FA1 风格  最大误差 = 4.440892098500626e-16
  FA2 风格  最大误差 = 4.440892098500626e-16

O 上的逐元素乘/除运算次数（n=32, M=B=8，共 4×4=16 个块对）：
                   逐元素乘法       逐元素除法
  FA1 风格            4096        2048
  FA2 风格            2048         512
  除法次数是 V2 的 4 倍，乘法多 1/3

三个结论：

1. **两种风格与标准 Attention 逐位一致**（1e-16 量级，纯浮点舍入）——“先归一化”与“后归一化”数学上完全等价；
2. FA1 的除法次数是 V2 的 **4 倍**（= 块对数 ÷ Q 块数 = 16 ÷ 4），乘法多 1/3——n 越大，差距按同比例放大；
3. 这些多出来的运算，GPU 上落在 CUDA core、昇腾上落在 **Vector 单元**——正是第 8 章“VF 优化”要精打细算的对象。

## 章节测验

1. FA1 概念算法的 外层 / 内层 分别循环谁？2.5 节伪代码（V2 结构）呢？
2. FA1 更新式中，O_i 为什么每步都要除以 l_new？“保持已归一化”的收益与代价各是什么？
3.（填空）FA1 的并行维度 = ____ × ____，因此在 ____ 场景下大量算力闲置。
4. 为什么逐元素（non-matmul）运算在 GPU 与 NPU 上都“不划算”？（提示：与哪类单元的速度差有关，以及“100 FLOPs / 字节”经验法则）
5.（判断）FA1 是近似算法，所以才能省显存。（对/错）
6. 本节实验中，块对数 16、Q 块数 4 时，FA1 与 V2 风格的除法次数分别是多少？比值为多少？
7. 按 FA1 的循环结构，K/V 与 Q/O/l/m 各被从 HBM 搬运多少轮？「O 的更新与 Q 绑定」的访存代价如何随 Tc 增长？

> 答案见 `answer/02.07_answer.txt`。

下一节：[2.8 FlashAttention V2](02.08_flash_attention_v2.ipynb)——三个手术，把 25%~40% 的硬件利用率提到 50%~73%。